In [137]:
import sys
from pathlib import Path

# Notebook lives in checkpoints/; project root is the parent.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn.functional as F

import config
from model import TinyTransformer
from generate_dataset import build_sequence
from measure_marginals import sample_psi_batch_multi

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N = config.N
print(f'N = {N},  VOCAB = {config.VOCAB},  SEQ_LEN = {config.SEQ_LEN}')
print(f'device: {DEVICE}')


N = 4,  VOCAB = 4,  SEQ_LEN = 16
device: cuda


In [138]:
# Pick a checkpoint to analyze. Re-run the notebook with a different
# CHECKPOINT to compare model_500 / model_1500 / model_2500 / model.pt.
CHECKPOINT = PROJECT_ROOT / 'checkpoints' / 'model_500.pt'

def load_model(path):
    m = TinyTransformer().to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    m.eval()
    return m

model = load_model(CHECKPOINT)
n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded {CHECKPOINT.name}  ({n_params:,} parameters)')

Loaded model_500.pt  (6,324,228 parameters)


/home/akash10/miniconda3/envs/aug-spm/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/tmp/ipykernel_682189/4237903001.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for

In [142]:
@torch.no_grad()                                                                                                            
def sample_psi_rejection(model, phis, max_attempts=50):
    model.eval()                                                                                                            
    N = config.N                                                                                                            
    if phis.dim() == 1:
        phis = phis.unsqueeze(0)                                                                                            
    B = phis.shape[0]           
    phis = phis.to(DEVICE)
    out     = torch.zeros((B, N), dtype=torch.long, device=DEVICE)                                                          
    pending = torch.ones(B,        dtype=torch.bool, device=DEVICE)
                                                                                                                            
    for _ in range(max_attempts):                                                                                           
        idx = torch.nonzero(pending, as_tuple=False).squeeze(1)
        if idx.numel() == 0:                                                                                                
            break                                                                                                           
        psis_try = sample_psi_unconstrained(model, phis[idx])  # (k, N) on DEVICE
        valid = is_perm_rows(psis_try, N)                      # (k,)   on DEVICE                                           
        out[idx[valid]] = psis_try[valid]                                                                                   
        pending[idx[valid]] = False                                                                                         
                                                                                                                            
    if pending.any():                                                                                                       
        raise RuntimeError(
            f"{int(pending.sum().item())} rows still invalid after {max_attempts} attempts; "
            "validity rate per phi too low — bump max_attempts or check the checkpoint."                                    
        )                                                                                                                   
    return out   
    
@torch.no_grad()
def sample_psi_unconstrained(model, phis, n=config.N):
    """AR sampling of psi WITHOUT the no-repeat mask. phis: (B, N).
    Returns (B, N) on CPU — possibly invalid (repeats allowed).
    """
    model.eval()
    if phis.dim() == 1:
        phis = phis.unsqueeze(0)
    B = phis.shape[0]
    psi_start = config.PSI.start                                  # = N

    psi_buf = torch.zeros((B, n), dtype=torch.long, device=DEVICE)
    seq = torch.cat([phis.to(DEVICE), psi_buf], dim=1)            # (B, 2N)

    for j in range(n):
        logits = model(seq)
        step_logits = logits[:, psi_start - 1 + j, :]
        probs = F.softmax(step_logits, dim=-1)
        sampled = torch.multinomial(probs, num_samples=1).squeeze(-1)
        seq[:, psi_start + j] = sampled
    return seq[:, psi_start:psi_start + n]

def is_perm_rows(samples, n):
    sorted_, _ = torch.sort(samples, dim=1)                                                                                 
    expected = torch.arange(n, device=samples.device).unsqueeze(0).expand_as(sorted_)
    return (sorted_ == expected).all(dim=1)  

def random_phis(B, n=N, seed=None):
    """B random permutations sampled uniformly from S_n."""
    g = torch.Generator()
    if seed is not None:
        g.manual_seed(seed)
    return torch.stack([torch.randperm(n, generator=g) for _ in range(B)])

def random_phis_with_constraint(B, j, u, n=N, seed=None):
      """B random permutations of S_n with perm[j] == u."""
      g = torch.Generator()                                                                                                   
      if seed is not None:
          g.manual_seed(seed)                                                                                                 
                  
      other_positions = torch.tensor([i for i in range(n) if i != j], dtype=torch.long)                                       
      other_values    = torch.tensor([v for v in range(n) if v != u], dtype=torch.long)
      n_other = n - 1                                                                                                         
  
      keys = torch.rand((B, n_other), generator=g)                                                                            
      perms = keys.argsort(dim=1)            # (B, n-1)  uniform random perms via argsort-of-random-keys
      perm_values = other_values[perms]      # (B, n-1)                                                                       
                  
      out = torch.zeros((B, n), dtype=torch.long)                                                                             
      out[:, j] = u
      out[:, other_positions] = perm_values                                                                                   
      return out  
def marginal_matrix(samples, n=config.N):
    """Build M[i, v] = fraction of `samples` with sample[i] == v.

    `samples` is a LongTensor of shape (K, N) of permutation values.
    """
    return F.one_hot(samples.long(), n).float().mean(dim=0)
    # --- old, equivalent (n^2 nested loops): -----------------------------
    # K = samples.shape[0]
    # M = torch.zeros(n, n)
    # for i in range(n):
    #     for v in range(n):
    #         M[i, v] = (samples[:, i] == v).float().mean()
    # return M
    # ---------------------------------------------------------------------


In [143]:
phis

tensor([[0, 1, 3, 2],
        [0, 1, 3, 2],
        [0, 1, 3, 2],
        ...,
        [0, 2, 1, 3],
        [0, 3, 2, 1],
        [0, 3, 1, 2]])

In [144]:
import random
import numpy as np

tau = {}

seed = 42
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

num_samples = 2**17
B=2**16
iters = num_samples//B

for j in range(N):
    for u in range(N):
        avg_mm = None
        for iter_val in range(iters):
            phis = random_phis_with_constraint(B, j=j, u=u)
            psis = sample_psi_rejection(model, phis)
            mm = marginal_matrix(psis)
            if avg_mm is None:
                avg_mm = mm
            else:
                avg_mm = 1/(iter_val+1)*mm + (iter_val/(iter_val+1))*avg_mm
        for i in range(N):
            for v in range(N):
                tau[(i, j, v, u)] = avg_mm[i, v]
# avg_mm


In [145]:
opts_max_spread = {}
opts_max_min_spread = {}
#j-> (i, v)
for j in range(N):
    scores_max_spread = torch.zeros((N, N))
    scores_max_min_spread = torch.zeros((N, N))
    for i in range(N):
        for v in range(N):
            u_func = torch.zeros((N,))
            for u in range(N):
                u_func[u] = tau[(i, j, v, u)]
            scores_max_spread[i, v] = torch.max(u_func) - torch.min(u_func)
            diffs = []
            for poss_1 in range(N):
                for poss_2 in range(poss_1+1, N):
                    diffs.append(abs(u_func[poss_1]-u_func[poss_2]))
            scores_max_min_spread[i, v] = min(diffs)
    i_ms, v_ms = (scores_max_spread.argmax() // N).item(), (scores_max_spread.argmax() %  N).item()                                                                   
    i_mm, v_mm = (scores_max_min_spread.argmax() // N).item(), (scores_max_min_spread.argmax() %  N).item()                                                               
                                                                                                                            
    opts_max_spread[j]     = (i_ms, v_ms)
    opts_max_min_spread[j] = (i_mm, v_mm)
            

In [146]:
from scipy.optimize import linear_sum_assignment
                                                                                                                            
def best_assignment(C):                                                                                                     
    """Returns (perm, total_cost) — the min-cost assignment."""                                                             
    C_np = C.cpu().numpy() if torch.is_tensor(C) else C                                                                     
    row_ind, col_ind = linear_sum_assignment(C_np)                                                                          
    perm = torch.tensor(col_ind, dtype=torch.long)
    cost = float(C_np[row_ind, col_ind].sum())                                                                              
    return perm, cost                                                                                                       
                                                                                                                                
import itertools                                                                                                            
                                                                                                                            
def top_k_assignments(C, k=None):
    """Top-k lowest-cost assignments. Returns list of (perm, cost), best first."""
    N = C.shape[0]                                                                                                          
    C_np = C.cpu().numpy() if torch.is_tensor(C) else C                                                                     
    rows = np.arange(N)                                                                                                     
    scored = [                                                                                                              
        (C_np[rows, list(perm)].sum(), perm)                                                                                
        for perm in itertools.permutations(range(N))
    ]                                                                                                                       
    scored.sort(key=lambda x: x[0])
    if k is not None:                                                                                                       
        scored = scored[:k]
    return [(torch.tensor(p, dtype=torch.long), float(c)) for c, p in scored]                                               


In [147]:
C_max_min = torch.zeros((N, N))
C_max = torch.zeros((N, N))
seed = 0
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
import itertools
B = 2**16
permutations = list(itertools.permutations(range(N)))

# trackers across all permutations
ranks_max_min, ranks_max = [], []
top1_mm = top1_m = topN_mm = topN_m = 0

for permutation in permutations[:]:
    permutation_tensor = torch.stack([torch.tensor(permutation, device='cuda' if torch.cuda.is_available() else 'cpu').long() for _ in range(B)], dim=0)
    psis = sample_psi_rejection(model, permutation_tensor)
    mm = marginal_matrix(psis)
    for j in range(N):
        i_j_max_min, v_j_max_min = opts_max_min_spread[j]
        i_j_max, v_j_max = opts_max_spread[j]
        p_j_max_min = mm[i_j_max_min, v_j_max_min]
        p_j_max = mm[i_j_max, v_j_max]
        for u in range(N):
            tau_max_min = tau[(i_j_max_min, j, v_j_max_min, u)]
            tau_max = tau[(i_j_max, j, v_j_max, u)]
            C_max_min[j, u] = torch.abs(tau_max_min - p_j_max_min)
            C_max[j, u] = torch.abs(tau_max - p_j_max)
    print(C_max_min)
    print(C_max)
    print(permutation)
    best_assignment_max_min = best_assignment(C_max_min)
    print(best_assignment_max_min)
    best_assignment_max = best_assignment(C_max)
    print(best_assignment_max)

    top_N_assignment_max_min = top_k_assignments(C_max_min, N)
    in_top_mm = any(torch.equal(torch.tensor(permutation, dtype=torch.long, device=pair[0].device), pair[0])
                    for pair in top_N_assignment_max_min)
    print(in_top_mm)
    print(top_N_assignment_max_min)

    top_N_assignment_max = top_k_assignments(C_max, N)
    in_top_m = any(torch.equal(torch.tensor(permutation, dtype=torch.long, device=pair[0].device), pair[0])
                   for pair in top_N_assignment_max)
    print(in_top_m)
    print(top_N_assignment_max)

    # rank of truth in the FULL sorted list (1 = Hungarian best, N! = worst)
    full_mm = top_k_assignments(C_max_min, k=None)
    full_m  = top_k_assignments(C_max,     k=None)
    truth_tup = tuple(permutation)
    rank_mm = next(i + 1 for i, (p, _) in enumerate(full_mm) if tuple(p.tolist()) == truth_tup)
    rank_m  = next(i + 1 for i, (p, _) in enumerate(full_m)  if tuple(p.tolist()) == truth_tup)
    print(f'rank of truth — max-min: {rank_mm}/{len(permutations)}, max: {rank_m}/{len(permutations)}')

    ranks_max_min.append(rank_mm)
    ranks_max.append(rank_m)
    top1_mm += int(rank_mm == 1)
    top1_m  += int(rank_m  == 1)
    topN_mm += int(in_top_mm)
    topN_m  += int(in_top_m)

    print('\n\n\n')

# ---- summary ----
T = len(permutations)
print('=' * 70)
print(f'Summary over all {T} permutations of S_{N}  '
      f'(random baselines: top-1 ≈ {100/T:.1f}%, top-{N} ≈ {100*N/T:.1f}%, mean rank ≈ {(T+1)/2:.1f})')
print('-' * 70)
print(f'{"":<20} {"top-1":>10} {"top-N":>10} {"mean rank":>14}')
print(f'{"max-min spread":<20} {top1_mm}/{T:<6} {topN_mm}/{T:<6} {sum(ranks_max_min)/T:>12.2f}')
print(f'{"max     spread":<20} {top1_m}/{T:<6} {topN_m}/{T:<6} {sum(ranks_max)/T:>12.2f}')

tensor([[0.0485, 0.0863, 0.2023, 0.1665],
        [0.1899, 0.0868, 0.0353, 0.0197],
        [0.1788, 0.0940, 0.0170, 0.1156],
        [0.1274, 0.2097, 0.0218, 0.0526]])
tensor([[0.0963, 0.0260, 0.0009, 0.3001],
        [0.2957, 0.2747, 0.0699, 0.4534],
        [0.0194, 0.1451, 0.0024, 0.1551],
        [0.0336, 0.1349, 0.3041, 0.1314]])
(0, 1, 2, 3)
(tensor([0, 3, 1, 2]), 0.18407440185546875)
(tensor([1, 2, 0, 3]), 0.246673583984375)
True
[(tensor([0, 3, 1, 2]), 0.18407440185546875), (tensor([0, 1, 2, 3]), 0.20484161376953125), (tensor([0, 2, 1, 3]), 0.23049163818359375), (tensor([1, 3, 2, 0]), 0.25032806396484375)]
False
[(tensor([1, 2, 0, 3]), 0.246673583984375), (tensor([1, 2, 3, 0]), 0.2845611572265625), (tensor([2, 1, 0, 3]), 0.42644500732421875), (tensor([0, 2, 1, 3]), 0.442718505859375)]
rank of truth — max-min: 2/24, max: 8/24




tensor([[0.3240, 0.1892, 0.4778, 0.1089],
        [0.1741, 0.0710, 0.0511, 0.0039],
        [0.2900, 0.0172, 0.0942, 0.2268],
        [0.4252, 0.0882,

In [148]:
tau[(1, 0, 3, 3)]

tensor(0.4652, device='cuda:0')

In [149]:
opts_max_min_spread

{0: (1, 3), 1: (0, 2), 2: (2, 3), 3: (3, 2)}

In [150]:
opts_max_spread

{0: (2, 2), 1: (0, 1), 2: (3, 0), 3: (1, 3)}

In [151]:
tau_opt = {}


In [152]:
scores_max_min_spread

tensor([[0.0037, 0.0191, 0.0207, 0.0120],
        [0.0610, 0.0099, 0.0203, 0.0034],
        [0.0358, 0.0160, 0.0115, 0.0432],
        [0.0541, 0.0120, 0.0744, 0.0156]])

In [153]:
scores_max_spread

tensor([[0.0905, 0.2933, 0.3293, 0.2755],
        [0.2099, 0.1875, 0.2211, 0.4390],
        [0.1332, 0.0798, 0.2752, 0.1949],
        [0.2494, 0.1807, 0.3370, 0.1314]])

In [154]:
tau[(0, 1, 1, 3)]

tensor(0.4596, device='cuda:0')

In [155]:
tensor([[0.2486, 0.3050, 0.1057, 0.3407],
        [0.1648, 0.3061, 0.3598, 0.1693],
        [0.2108, 0.1654, 0.3079, 0.3159],
        [0.3758, 0.2235, 0.2266, 0.1741]], device='cuda:0')

tensor([[0.2307, 0.2820, 0.2073, 0.2801],
        [0.1150, 0.0663, 0.3905, 0.4282],
        [0.3992, 0.2017, 0.2586, 0.1405],
        [0.2557, 0.4491, 0.1438, 0.1514]], device='cuda:0')

tensor([[0.2501, 0.0766, 0.3329, 0.3404],
        [0.1824, 0.2924, 0.2515, 0.2737],
        [0.3492, 0.3122, 0.1516, 0.1870],
        [0.2183, 0.3187, 0.2640, 0.1989]], device='cuda:0')

tensor([[0.1884, 0.4594, 0.2773, 0.0749],
        [0.1807, 0.1852, 0.3082, 0.3259],
        [0.3372, 0.1764, 0.1204, 0.3660],
        [0.2937, 0.1790, 0.2941, 0.2331]], device='cuda:0')

NameError: name 'tensor' is not defined

In [ ]:
#marginal_matrix(psis)[0][1] corresponds to prob(psis[0]=1)

tensor(0.2715, device='cuda:0')

In [ ]:
marginal_matrix[0][1]

TypeError: 'function' object is not subscriptable

In [ ]:
psis==1

tensor([[False, False, False,  True],
        [ True, False, False, False],
        [ True, False, False, False],
        ...,
        [False, False, False,  True],
        [False, False, False,  True],
        [ True, False, False, False]], device='cuda:0')

In [ ]:
torch.sum(psis==1, axis=0)/psis.shape[0]

tensor([0.2715, 0.0674, 0.2178, 0.4414], device='cuda:0')